In [93]:
!pip show torch torchmetrics --quiet
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch: 2.10.0+cu128
CUDA available: True


In [94]:
from pytorch_tabular.models import FTTransformerConfig
help(FTTransformerConfig.__init__)

Help on function __init__ in module pytorch_tabular.models.ft_transformer.config:

__init__(self, task: str, head: Optional[str] = 'LinearHead', head_config: Optional[Dict] = <factory>, embedding_dims: Optional[List] = None, embedding_dropout: float = 0.0, batch_norm_continuous_input: bool = True, learning_rate: float = 0.001, loss: Optional[str] = None, metrics: Optional[List[str]] = None, metrics_prob_input: Optional[List[bool]] = None, metrics_params: Optional[List] = None, target_range: Optional[List] = None, virtual_batch_size: Optional[int] = None, seed: int = 42, _module_src: str = 'models.ft_transformer', _model_name: str = 'FTTransformerModel', _backbone_name: str = 'FTTransformerBackbone', _config_name: str = 'FTTransformerConfig', input_embed_dim: int = 32, embedding_initialization: Optional[str] = 'kaiming_uniform', embedding_bias: bool = True, share_embedding: bool = False, share_embedding_strategy: Optional[str] = 'fraction', shared_embedding_fraction: float = 0.25, attn_

In [95]:
!pip install pytorch-tabular --quiet

In [96]:
### Cell 2 — Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.metrics import mean_squared_error
sns.set(style="darkgrid")

In [97]:
### Cell 3 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
### Cell 4 — Load Data
def read_file(filename):
    directory = '/content/drive/My Drive/210_capstone/final_datasets/full_hist_attr/'
    return pd.read_parquet(directory + filename + '.parquet')

In [99]:
df = read_file('full_data_18to23_asofFeb25_1')

In [110]:
### Cell 5 — Rename Columns
df = df.rename(columns={
    'Date':  'date',
    'County': 'county',
    'BEV':   'bev',
    'PHEV':  'phev',
    'FCEV':  'fcev',
    'Max_Daily_Electricity_Usage': 'max_daily_electricity',
    'Per_Capita_Personal_Income_Adjusted': 'per_cap_income',
    'Total': 'total_ev_charge'
})

In [111]:
### Cell 6 — Target Engineering (BEFORE split)
df['date'] = pd.to_datetime(df['date'])
df['max_elec_per_capita']     = df['max_daily_electricity'] / df['total_pop']
df['max_elec_per_capita_log'] = np.log(df['max_elec_per_capita'])

target = 'max_elec_per_capita_log'

print(df.shape)


(127078, 82)


In [102]:
# Add to your notebook after Cell 6 (target engineering)

# Compute 2018 baseline MW per capita per county
baseline_2018 = (df[df['date'].dt.year == 2018]
    .groupby('county')
    .apply(lambda x: (x['max_daily_electricity'] / x['total_pop']).mean(),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'baseline_mw_per_capita_2018'}))

#print(baseline_2018.sort_values('baseline_mw_per_capita_2018', ascending=False))

# Merge into main df
df = df.merge(baseline_2018, on='county', how='left')

In [103]:
ROLLING_SPECS = [
    ("cdd65_pop_roll5",       "cdd65_pop",  5,  "sum"),
    ("hdd65_pop_roll5",       "hdd65_pop",  5,  "sum"),
    ("tmax_k_pop_roll5_max",  "tmax_k_pop", 5,  "max"),
    ("tmax_k_pop_roll7_mean", "tmax_k_pop", 7,  "mean"),

]

def add_rolling_features(df):
    df = df.sort_values(["county", "date"]).copy()
    g = df.groupby("county", sort=False)
    for new_col, src_col, window, agg in ROLLING_SPECS:
        df[new_col] = g[src_col].transform(
            lambda x, w=window, a=agg: getattr(x.rolling(w, min_periods=1), a)()
        )
    return df

In [104]:
# Compute 2019 mobility baseline before dropping the columns
mobility_baseline = (
    df[df['date'].dt.year == 2019]
    .groupby(['county', 'day_of_week'])[['staying_total', 'entering_total', 'leaving_total']]
    .median()
    .reset_index()
)
print("Mobility baseline shape:", mobility_baseline.shape)


Mobility baseline shape: (406, 5)


In [105]:
df = add_rolling_features(df)

# Replace mobility with frozen 2019 baseline
df = df.drop(columns=['staying_total', 'entering_total', 'leaving_total'])
df = df.merge(mobility_baseline, on=['county', 'day_of_week'], how='left')


In [106]:
### Dew Point Depression Feature
import numpy as np

def add_dpd_features(df):
    P = 101325  # standard pressure Pa
    w = df['spfh_peak_kgkg_pop']

    # specific humidity → vapor pressure
    e = (w * P) / (0.622 + w)

    # vapor pressure → dew point (K)
    dpt_c = 243.04 * np.log(e / 611.2) / (17.625 - np.log(e / 611.2))
    df['dpt_derived_k'] = dpt_c + 273.15

    # dew point depression (K) — large = dry, small = humid
    df['dpd_k'] = (df['tmax_k_pop'] - df['dpt_derived_k']).clip(lower=0)

    # rolling 5-day mean depression
    df = df.sort_values(['county', 'date'])
    df['dpd_k_roll5'] = (df
        .groupby('county')['dpd_k']
        .transform(lambda x: x.rolling(5, min_periods=1).mean()))

    return df

df = add_dpd_features(df)

print(f"dpd_k stats:\n{df['dpd_k'].describe()}")

dpd_k stats:
count    127078.000000
mean         15.181825
std           8.168411
min           0.000000
25%           8.577026
50%          14.242157
75%          21.110504
max          52.180115
Name: dpd_k, dtype: float64


In [107]:
# Impute per_cap_income — county median, fall back to global median
global_median = df['per_cap_income'].median()

df['per_cap_income'] = (df.groupby('county')['per_cap_income']
    .transform(lambda x: x.fillna(x.median())))

df['per_cap_income'] = df['per_cap_income'].fillna(global_median)

print(df['per_cap_income'].isna().sum())  # should be 0

0


# THE SPLIT

In [108]:
# Split
train_df = df[df['date'].dt.year <= 2021].copy()
val_df   = df[df['date'].dt.year == 2022].copy()
test_df  = df[df['date'].dt.year == 2023].copy()

print(f"Train: {train_df.shape}  ({train_df['date'].dt.year.min()}–{train_df['date'].dt.year.max()})")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")


Train: (84738, 82)  (2018–2021)
Val:   (21170, 82)
Test:  (21170, 82)


In [112]:
global_median = train_df['per_cap_income'].median()

numeric_cont_cols = [c for c in cont_cols if train_df[c].dtype != 'object']

for col in numeric_cont_cols:
    train_df[col] = train_df[col].fillna(global_median).astype('float32')
    val_df[col] = val_df[col].fillna(global_median).astype('float32')

print("NaNs in train_df:", train_df[numeric_cont_cols].isna().sum().sum())
print("NaNs in val_df:", val_df[numeric_cont_cols].isna().sum().sum())

NaNs in train_df: 0
NaNs in val_df: 0


In [126]:
selected_features = [
    'county', 'day_of_week', 'month', 'is_holiday',
    'tmax_k_pop', 'tmin_k_pop',
    'cdd65_pop_roll5', 'hdd65_pop_roll5',
    'spfh_peak_kgkg_pop', 'dpd_k',
    'cuml_dc_load', 'bev',
    'baseline_mw_per_capita_2018', 'total_pop',
]

In [113]:
# ### Cell 8 — Features
# selected_features = [
#     # CATEGORICAL
#     "county",
#     "day_of_week",
#     # CALENDAR
#     "quarter",
#     "month",
#     "is_holiday",
#     # TEMPERATURE
#     "tmax_k_pop",
#     "tmin_k_pop",
#     "trange_k",
#     # DEGREE DAYS
#     "hdd65_pop",
#     "cdd65_pop",
#     "cdd75_pop",
#     # ROLLING WEATHER
#     "cdd65_pop_roll5",
#     "hdd65_pop_roll5",
#     "tmax_k_pop_roll5_max",
#     "tmax_k_pop_roll7_mean",
#     # HUMIDITY
#     "spfh_peak_kgkg_pop",
#     # WIND
#     "wind_peak_ms_pop",
#     # MOBILITY
#     # "staying_total",
#     # "entering_total",
#     # "leaving_total",
#     # DATA CENTERS
#     "cuml_count",
#     "cuml_sq_foot",
#     "cuml_utility_cap",
#     "cuml_dc_load",
#     # EVs
#      "bev",
#     #   "phev",
#     #  "fcev",
#       # "total_ev_pop",
#     # SOCIOECONOMIC
#     "per_cap_income",
#     "total_pop",
#     ## MW / county / pop
#     "baseline_mw_per_capita_2018",
#     ##dpd_k
#     'dpd_k',
#     'dpd_k_roll5',
# ]

# cat_cols = ["county", "day_of_week"]
# print(f"Features: {len(selected_features)}")

Features: 27


In [ ]:
# selected_features=["month", "total_pop", "est_median_income", "cuml_utility_cap", "cdd65_pop", "hdd65_pop", "spfh_peak_kgkg_pop", "cloud_cover_pct_pop", "wind_low_ms_pop", "wind_peak_ms_pop", "total_ev_pop"]

In [ ]:
# # Check for NaNs in training features
# cont_cols = [f for f in selected_features if f not in ['day_of_week']]

# print("NaNs in train_df:")
# print(train_df[cont_cols].isna().sum()[train_df[cont_cols].isna().sum() > 0])

# print("\nNaNs in val_df:")
# print(val_df[cont_cols].isna().sum()[val_df[cont_cols].isna().sum() > 0])

# print("\nInfs in train_df:")
# print(np.isinf(train_df[cont_cols]).sum()[np.isinf(train_df[cont_cols]).sum() > 0])

In [127]:
cat_cols = ["county", "day_of_week"]
print(f"Features: {len(selected_features)}")

Features: 14


In [128]:
import shutil
shutil.rmtree('/content/saved_models', ignore_errors=True)

In [129]:
from pytorch_tabular import TabularModel
from pytorch_tabular.models import FTTransformerConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

# ── Data Config ───────────────────────────────────────────────────────────────
data_config = DataConfig(
    target=['max_elec_per_capita_log'],
    continuous_cols=[f for f in selected_features
                     if f not in ['county', 'day_of_week']],
    categorical_cols=['county', 'day_of_week'],
)

# ── Model Config ──────────────────────────────────────────────────────────────
model_config = FTTransformerConfig(
    task='regression',
    learning_rate=1e-3,
    num_attn_blocks=3,
    num_heads=8,
    input_embed_dim=64,
    attn_dropout=0.15,
    ff_dropout=0.15,
    add_norm_dropout=0.15,
)


# ── Trainer Config ────────────────────────────────────────────────────────────
trainer_config = TrainerConfig(
    batch_size=512,        # smaller batch = more gradient updates, better generalization
    max_epochs=100,
    early_stopping='valid_loss',
    early_stopping_patience=15,
    checkpoints='valid_loss',
    load_best=True,
    accelerator='gpu',
    devices=1,
)

# ── Optimizer Config ──────────────────────────────────────────────────────────

optimizer_config = OptimizerConfig(
    optimizer='AdamW',
    optimizer_params={'weight_decay': 1e-3},
    lr_scheduler='ReduceLROnPlateau',
    lr_scheduler_params={'patience': 5, 'factor': 0.5},
    lr_scheduler_monitor_metric='valid_loss',
)

# ── Build & Train ─────────────────────────────────────────────────────────────
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

tabular_model.fit(
    train=train_df,
    validation=val_df,
)

2026-03-02 05:35:38,146 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-03-02 05:35:38,165 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-02 05:35:38,226 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-02 05:35:38,353 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-03-02 05:35:38,398 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Exp

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ FTTransformerBackbone │  541 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding2dLayer      │  6.0 K │ train │     0 │
│ 2 │ _head            │ LinearHead            │     65 │ train │     0 │
│ 3 │ loss             │ MSELoss               │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 547 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 547 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 72                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

2026-03-02 05:40:02,073 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-02 05:40:02,074 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


In [130]:
val_df['actual_mwh'] = np.exp(val_df[target]) * val_df['total_pop']

preds_ft = tabular_model.predict(val_df)
val_df['preds_mwh_ft'] = np.exp(preds_ft['max_elec_per_capita_log_prediction']) * val_df['total_pop']

ft_rmse = np.sqrt(mean_squared_error(val_df['actual_mwh'], val_df['preds_mwh_ft']))

w = val_df['total_pop'].values
wrmse = np.sqrt(np.sum(w * (val_df['actual_mwh'] - val_df['preds_mwh_ft'])**2) / np.sum(w))
wmean = np.average(val_df['actual_mwh'], weights=w)
pop_wtd = 100 * wrmse / wmean

print(f"FT-Transformer:")
print(f"  RMSE MWh:        {ft_rmse:,.0f}")
print(f"  Pop-wtd RMSE %:  {pop_wtd:.1f}%")
print(f"\nLightGBM v4 baseline:")
print(f"  RMSE MWh:        144")
print(f"  Pop-wtd RMSE %:  12.2%")

FT-Transformer:
  RMSE MWh:        181
  Pop-wtd RMSE %:  14.2%

LightGBM v4 baseline:
  RMSE MWh:        144
  Pop-wtd RMSE %:  12.2%


In [120]:
print(f"Best epoch: {tabular_model.trainer.checkpoint_callback.best_model_path}")

Best epoch: /content/saved_models/regression-6_epoch=14-valid_loss=0.03.ckpt


In [131]:
# Get attention importance matrix
importance_df = tabular_model.feature_importance()
print(importance_df)

                       Features  importance
0                        county    0.065778
1                   day_of_week    0.069205
2                         month    0.070817
3                    is_holiday    0.070456
4                    tmax_k_pop    0.070201
5                    tmin_k_pop    0.070924
6               cdd65_pop_roll5    0.069715
7               hdd65_pop_roll5    0.070230
8            spfh_peak_kgkg_pop    0.069668
9                         dpd_k    0.070684
10                 cuml_dc_load    0.067751
11                          bev    0.072210
12  baseline_mw_per_capita_2018    0.068624
13                    total_pop    0.067775
